# cuconv on a Colab T4

Runs the whole repo end to end: build the kernels, check every stage against cuDNN, benchmark, profile, draw the chart and the feature-map grid.

Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**. About ten minutes. The last cell downloads a zip of `results/<label>/` (numbers as JSON, both figures, the full log).

In [ ]:
!nvidia-smi
!nvcc --version | tail -2
import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'cudnn', torch.backends.cudnn.version())

In [ ]:
import os
if not os.path.exists('/content/cuconv'):
    !git clone --quiet https://github.com/noorps/cuconv.git /content/cuconv
%cd /content/cuconv
!git pull --quiet; git log --oneline -1

In [ ]:
import time, os
LABEL = time.strftime('t4-%Y%m%d-%H%M')
os.makedirs(f'results/{LABEL}', exist_ok=True)
print(LABEL)
!bash run.sh $LABEL 2>&1 | tee results/$LABEL/run.log

In [ ]:
from IPython.display import Image, display
for f in ('gflops.png', 'featuremaps.png'):
    p = f'results/{LABEL}/{f}'
    if os.path.exists(p):
        display(Image(p, width=1100))
    else:
        print('missing', p, '- see the log above')

In [ ]:
import json
p = f'results/{LABEL}/bench.json'
if os.path.exists(p):
    b = json.load(open(p))
    print(json.dumps(b['device'], indent=1))
    for r in b['rows']:
        print(f"{r['shape']:<26}{r['stage']:<10}{r['gflops']:>8.1f} GFLOP/s  [{r['gflops_min']:.1f}-{r['gflops_max']:.1f}]  {r['pct_cudnn']:6.1f}% of cuDNN")

In [ ]:
# Zip the results and offer them two ways: a browser download, and a link served
# through Colab's port proxy (works when the browser blocks repeated downloads).
import shutil, threading, http.server, functools
shutil.make_archive(f'/content/{LABEL}', 'zip', '/content/cuconv/results', LABEL)
handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory='/content')
threading.Thread(target=http.server.ThreadingHTTPServer(('127.0.0.1', 8000), handler).serve_forever, daemon=True).start()
from google.colab import output
print(output.eval_js('google.colab.kernel.proxyPort(8000)') + f'{LABEL}.zip')
from google.colab import files
files.download(f'/content/{LABEL}.zip')